# LEGO GINN Model Test

This notebook tests a trained LEGO GINN model by:
1. Loading a checkpoint
2. Extracting meshes for different brick types
3. Visualizing the results

**Key fix**: Uses correct bounds per brick type (1x2 vs 1x4 have different sizes)

In [1]:
# Setup
import sys
sys.path.append('..')

import torch
import numpy as np
import glob
import os
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [2]:
# Load checkpoint
from util.checkpointing import load_yaml_and_drop_keys
from util.misc import get_model, get_problem
from models.net_w_partials import NetWithPartials

# Find latest checkpoint (include path/to/dir/cond_wire where training saves)
ckpt_patterns = [
    'path/to/dir/cond_wire/**/*-model.pt',
    '../path/to/dir/cond_wire/**/*-model.pt',
    '../checkpoints/cond_wire/**/*-model.pt',
    '../checkpoints/**/*-model.pt',
]

ckpt_path = None
for pattern in ckpt_patterns:
    candidates = sorted(glob.glob(pattern, recursive=True), key=os.path.getmtime)
    if candidates:
        ckpt_path = candidates[-1]
        break

if ckpt_path is None:
    raise FileNotFoundError("No checkpoint found!")
# Optional: point to a specific run (must be the .pt FILE). Forward slashes work from any cwd.
#ckpt_path = "path/to/dir/cond_wire/2026_02_03__22_28_55-89jdkw55/2026_02_03__22_28_55-89jdkw55-model.pt"
# Resolve path when running from notebooks/ (path is relative to repo root)
if not os.path.exists(ckpt_path) and not os.path.isabs(ckpt_path):
    alt = os.path.normpath(os.path.join('..', ckpt_path))
    if os.path.exists(alt):
        ckpt_path = alt
if not os.path.exists(ckpt_path):
    raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")
print(f"Loading: {ckpt_path}")
ckpt = torch.load(ckpt_path, map_location='cpu')
print(f"Checkpoint keys: {list(ckpt.keys())}")

Loading: ../path/to/dir/cond_wire\2026_02_05__02_02_38-3k34l224\2026_02_05__02_02_38-3k34l224-model.pt
Checkpoint keys: ['state_dict', 'init_params']


C:\Users\chukw\AppData\Local\Temp\ipykernel_40804\996577723.py:33: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location='cpu')


In [3]:
# Extract config and model args (load run config if checkpoint has none; use init_params when present)
import yaml
config = ckpt.get('config', {})
if not config and 'ckpt_path' in dir() and ckpt_path:
    run_dir = os.path.dirname(ckpt_path)
    config_files = glob.glob(os.path.join(run_dir, '*-config.yml'))
    if config_files:
        with open(config_files[0], 'r') as f:
            config = yaml.safe_load(f)
        print(f"Loaded config from run dir: {config_files[0]}")
init_params = ckpt.get('init_params', {})
if init_params:
    model_args = {k: v for k, v in init_params.items() if k not in ('layers',)}
    raw_layers = init_params.get('layers', [128, 128, 128])
    # get_model() does layers.insert(0, input_size) and layers.append(1), so pass hidden dims only
    if len(raw_layers) >= 2 and (raw_layers[-1] == 1 or (hasattr(raw_layers[-1], 'item') and raw_layers[-1].item() == 1)):
        model_args['layers'] = raw_layers[1:-1].copy()  # drop input dim and output dim
        # Infer nz from checkpoint input dim: input_dim = nx + nz + 2*(tiled) + 1*(dist)
        input_dim = int(raw_layers[0]) if hasattr(raw_layers[0], '__int__') else raw_layers[0]
        tiled = 2 if model_args.get('use_tiled_coords', False) else 0
        dist = 1 if (model_args.get('use_dist_to_edge', False) and model_args.get('n_studs_values')) else 0
        model_args['nz'] = input_dim - 3 - tiled - dist  # nx=3
    else:
        model_args['layers'] = list(raw_layers)
    model_args.setdefault('model_str', 'cond_wire')
    model_args.setdefault('nx', 3)
    model_args.setdefault('nz', init_params.get('nz', 2))
    if init_params.get('n_studs_values'):
        config['n_studs_values'] = init_params['n_studs_values']
        config['condition_on_n_studs'] = len(init_params['n_studs_values']) > 1
    if 'problem' not in config or not config.get('problem'):
        config['problem'] = config.get('problem') or {'problem_str': 'lego_1xN', 'n_studs': 4}
    model_args['w0_initial'] = model_args.pop('first_omega_0', 30.0)
    model_args['w0'] = model_args.pop('hidden_omega_0', 30.0)
    model_args['wire_scale'] = model_args.pop('scale', 10.0)
else:
    model_args = config.get('model', {})
print("Config keys:", list(config.keys()))
print("Model args (from init_params or config):", {k: v for k, v in list(model_args.items())[:15]})

# Key parameters
nz = model_args.get('nz', 3)
nx = model_args.get('nx', 3)
n_studs_values = config.get('n_studs_values', [2, 4])
# Include 1x5 if not already present (optional)
if 5 not in n_studs_values:
    n_studs_values = list(n_studs_values) + [5]
# Override to try different lengths: uncomment and set e.g. [2, 4, 5] or [2, 3, 4, 5, 6]
n_studs_values = [2,4,5,8,12]
condition_on_n_studs = config.get('condition_on_n_studs', False)

print(f"\nnz={nz}, nx={nx}")
print(f"n_studs_values={n_studs_values}")
print(f"condition_on_n_studs={condition_on_n_studs}")

Loaded config from run dir: ../path/to/dir/cond_wire\2026_02_05__02_02_38-3k34l224\2026_02_05__02_02_38-3k34l224-config.yml
Config keys: ['FEM', 'adaptive_penalty', 'args_str', 'condition_on_height', 'condition_on_n_studs', 'condition_on_n_studs_y', 'connectivity_grad_tol', 'connectivity_n_iter', 'connectivity_n_samples', 'connectivity_update_every_n_epochs', 'cuboid_boundary_weight', 'cuboid_n_boundary', 'cuboid_n_corners', 'cuboid_n_far', 'cuboid_n_inside', 'cuboid_n_near_faces', 'cuboid_safety_margin', 'curvature_expression', 'curvature_pts_source', 'curvature_use_gradnorm_weights', 'data', 'decay_steps', 'dirichlet_use_surface_points', 'div_neighbor_agg_fn', 'div_norm_order', 'diversity_aggregation', 'diversity_pts_source', 'diversity_type', 'exclude_surface_points_close_to_interface_cutoff', 'field_losses', 'ginn_bsize', 'gpu_list', 'grad_clipping', 'height_values', 'lambda_chamfer_div', 'lambda_comp', 'lambda_connectivity', 'lambda_cuboid_primitive', 'lambda_cuboid_rule', 'lambda

In [4]:
# Create model and load weights
# First, inspect state_dict to determine actual model architecture
state_dict = ckpt.get('model', ckpt.get('state_dict', None))
if state_dict:
    print("State dict keys:")
    for k in state_dict.keys():
        print(f"  {k}: {state_dict[k].shape}")

# Detect model type from state_dict keys
has_freqs_scale = any('freqs' in k for k in state_dict.keys())
has_freq_scale = any('freq_scale' in k for k in state_dict.keys())

print(f"\nhas_freqs_scale (legacy): {has_freqs_scale}")
print(f"has_freq_scale (new): {has_freq_scale}")

# Infer layer widths from state_dict BEFORE conversion
# Legacy format: freqs.weight shape is [hidden_dim, input_dim]
layer_widths = []
if has_freqs_scale:
    # Find all freqs.weight keys to get hidden dims
    for i in range(10):  # max 10 layers
        key = f'net.{i}.freqs.weight'
        if key in state_dict:
            hidden_dim = state_dict[key].shape[0]
            layer_widths.append(hidden_dim)
        else:
            break
    print(f"Inferred layer widths from legacy format: {layer_widths}")

# If legacy format, we need to convert state_dict keys for ConditionalWIRE
if has_freqs_scale and not has_freq_scale:
    print("\nConverting legacy WIRE state_dict to new format...")
    new_state_dict = {}
    
    for k, v in state_dict.items():
        if '.freqs.' in k:
            layer_idx = k.split('.')[1]
            param_type = k.split('.')[-1]  # weight or bias
            new_state_dict[f'_freqs_{layer_idx}_{param_type}'] = v
        elif '.scale.' in k:
            layer_idx = k.split('.')[1]
            param_type = k.split('.')[-1]
            new_state_dict[f'_scale_{layer_idx}_{param_type}'] = v
        else:
            new_state_dict[k] = v
    
    # Now combine freqs and scale into freq_scale
    final_state_dict = {}
    layer_indices = set()
    for k in new_state_dict.keys():
        if k.startswith('_freqs_'):
            layer_indices.add(k.split('_')[2])
    
    for k, v in new_state_dict.items():
        if k.startswith('_freqs_') or k.startswith('_scale_'):
            continue
        final_state_dict[k] = v
    
    for layer_idx in sorted(layer_indices, key=int):
        for param_type in ['weight', 'bias']:
            freqs_key = f'_freqs_{layer_idx}_{param_type}'
            scale_key = f'_scale_{layer_idx}_{param_type}'
            if freqs_key in new_state_dict and scale_key in new_state_dict:
                # Concatenate freqs and scale tensors along output dimension
                combined = torch.cat([new_state_dict[freqs_key], new_state_dict[scale_key]], dim=0)
                final_state_dict[f'net.{layer_idx}.freq_scale.{param_type}'] = combined
                print(f"  Combined net.{layer_idx}.freq_scale.{param_type}: {combined.shape}")
    
    state_dict = final_state_dict
    print(f"\nConverted state dict keys: {list(state_dict.keys())}")

# Always use cond_wire since that's what's in this codebase
model_str = 'cond_wire'

# Ensure model_args has required fields  
model_args['model_str'] = model_str
if 'nx' not in model_args:
    model_args['nx'] = nx
if 'nz' not in model_args:
    model_args['nz'] = nz

# Use inferred layer widths only when checkpoint has no init_params (else keep architecture from init_params)
if init_params and model_args.get('layers'):
    print(f"Using layers from init_params: {model_args['layers']}")
elif layer_widths:
    model_args['layers'] = layer_widths
    print(f"Using inferred layers: {layer_widths}")
elif 'layers' not in model_args:
    model_args['layers'] = [256, 256, 256, 256]

if 'return_density' not in model_args:
    model_args['return_density'] = config.get('nf_is_density', False)

# CRITICAL: omega0/sigma0 are non-trainable hyperparams - must be set on model creation
# get_model uses: w0_initial -> first_omega_0, w0 -> hidden_omega_0, wire_scale -> scale
# Remove any conflicting keys and use correct names for get_model
for key in ['first_omega_0', 'hidden_omega_0', 'scale']:
    model_args.pop(key, None)

if 'w0_initial' not in model_args:
    model_args['w0_initial'] = config.get('w0_initial', config.get('first_omega_0', 30.0))
if 'w0' not in model_args:
    model_args['w0'] = config.get('w0', config.get('hidden_omega_0', 30.0))
if 'wire_scale' not in model_args:
    model_args['wire_scale'] = config.get('wire_scale', config.get('sigma0', 10.0))

print(f"\nFinal model_args: {model_args}")

model = get_model(**model_args)
print(f"Model type: {type(model).__name__}")

# Debug: check that omega_0 and scale_0 are set on layers
for i, layer in enumerate(model.net):
    if hasattr(layer, 'omega_0'):
        print(f"  Layer {i}: omega_0={layer.omega_0}, scale_0={layer.scale_0}")

# Debug: print model state dict keys for comparison
print("\nModel expects these keys:")
for k, v in model.state_dict().items():
    print(f"  {k}: {v.shape}")

# Load weights
model.load_state_dict(state_dict)
print("\nLoaded model weights successfully!")

model.eval()
model.to(device)

# Create NetWithPartials wrapper
netp = NetWithPartials.create_from_model(model, nz=model_args['nz'], nx=model_args['nx'])
netp.params = {k: v.to(device) for k, v in netp.params.items()}
print("Model ready")

State dict keys:
  net.0.freq_scale.weight: torch.Size([512, 11])
  net.0.freq_scale.bias: torch.Size([512])
  net.1.freq_scale.weight: torch.Size([512, 256])
  net.1.freq_scale.bias: torch.Size([512])
  net.2.freq_scale.weight: torch.Size([512, 256])
  net.2.freq_scale.bias: torch.Size([512])
  net.3.freq_scale.weight: torch.Size([512, 256])
  net.3.freq_scale.bias: torch.Size([512])
  net.4.weight: torch.Size([1, 256])
  net.4.bias: torch.Size([1])

has_freqs_scale (legacy): False
has_freq_scale (new): True
Using layers from init_params: [256, 256, 256, 256]

Final model_args: {'return_density': False, 'use_legacy_gabor': False, 'use_tiled_coords': True, 'stud_spacing_y': 1.0, 'use_dist_to_edge': True, 'n_studs_values': [2, 4, 5, 8, 12], 'n_norm_col': 2, 'use_hypernet': False, 'c_dim': 3, 'layers': [256, 256, 256, 256], 'nz': 5, 'model_str': 'cond_wire', 'nx': 3, 'w0_initial': 18, 'w0': 1.0, 'wire_scale': 6}
Model type: ConditionalWIRE
  Layer 0: omega_0=18, scale_0=6
  Layer 1: omeg

In [5]:
# Create problems for each brick type - CRITICAL for correct bounds
# no_studs=False so bounds include stud height; mesh will show studs only if checkpoint was trained with studs
problems = {}
base_problem_cfg = config.get('problem', {'problem_str': 'lego_1xN', 'n_studs': 4})

# Get sampling config - provide defaults if missing
problem_sampling = config.get('problem_sampling', {})
default_sampling = {
    'nx': 3,
    'n_points_envelope': 1000,
    'n_points_interfaces': 1000, 
    'n_points_domain': 1000,
}
for k, v in default_sampling.items():
    if k not in problem_sampling:
        problem_sampling[k] = v

print(f"Problem sampling: {problem_sampling}")

for n in n_studs_values:
    prob_cfg = {**base_problem_cfg, 'n_studs': n, 'n_studs_y': None, 'height_scale': 1.0, 'no_studs': False}
    # Remove 'problem_str' from sampling if it exists
    sampling = {k: v for k, v in problem_sampling.items() if k != 'problem_str'}
    prob = get_problem(problem_config=prob_cfg, **sampling)
    problems[n] = prob
    print(f"\n1x{n} brick bounds:")
    print(f"  x: [{prob.bounds[0,0]:.3f}, {prob.bounds[0,1]:.3f}]")
    print(f"  y: [{prob.bounds[1,0]:.3f}, {prob.bounds[1,1]:.3f}]")
    print(f"  z: [{prob.bounds[2,0]:.3f}, {prob.bounds[2,1]:.3f}]")

Problem sampling: {'n_points_domain': 2048, 'n_points_envelope': 8192, 'n_points_interfaces': 10000, 'n_points_normals': 10000, 'n_points_surface': 20000, 'nx': 3}
Created LEGO 1x2 problem (height_scale=1.00):
  Bounds: [[-0.987500011920929, 0.987500011920929], [-0.48750001192092896, 0.48750001192092896], [-0.706250011920929, 0.706250011920929]]
  Stud centers: 2 studs
  Points: 8192 far_outside, 16384 outside, 8192 around_if, 4096 inside, 9998 interface, 6 walls

1x2 brick bounds:
  x: [-0.988, 0.988]
  y: [-0.488, 0.488]
  z: [-0.706, 0.706]
Created LEGO 1x4 problem (height_scale=1.00):
  Bounds: [[-1.9874999523162842, 1.9874999523162842], [-0.48750001192092896, 0.48750001192092896], [-0.706250011920929, 0.706250011920929]]
  Stud centers: 4 studs
  Points: 8192 far_outside, 16384 outside, 8192 around_if, 4096 inside, 9996 interface, 6 walls

1x4 brick bounds:
  x: [-1.987, 1.987]
  y: [-0.488, 0.488]
  z: [-0.706, 0.706]
Created LEGO 1x5 problem (height_scale=1.00):
  Bounds: [[-2.4

In [6]:
# Helper: Create latent vector with conditioning
def make_latent(z_base, n_studs, config):
    """
    Create full latent vector with conditioning.
    
    Args:
        z_base: base latent (2D tensor or list)
        n_studs: number of studs (e.g., 2 for 1x2)
        config: full config dict
    """
    if isinstance(z_base, list):
        z_base = torch.tensor(z_base, dtype=torch.float32)
    z_base = z_base.to(device)
    
    n_vals = config.get('n_studs_values', [2, 2])
    n_min, n_max = min(n_vals), max(n_vals)
    
    # Build conditioning columns
    cond = []
    if config.get('condition_on_n_studs', False):
        n_norm = (n_studs - n_min) / max(n_max - n_min, 1)
        cond.append(n_norm)
    if config.get('condition_on_n_studs_y', False):
        cond.append(0.0)  # ny=1 → normalized to 0
    if config.get('condition_on_height', False):
        cond.append(0.0)  # h=1.0 → normalized to 0 if only one height
    
    if cond:
        cond_tensor = torch.tensor(cond, dtype=torch.float32, device=device)
        z_full = torch.cat([z_base, cond_tensor])
    else:
        z_full = z_base
    
    return z_full

# Test
z_test = make_latent([0.05, 0.05], n_studs=2, config=config)
print(f"Latent for 1x2: {z_test}")
z_test = make_latent([0.05, 0.05], n_studs=2, config=config)
print(f"Latent for 1x4: {z_test}")

Latent for 1x2: tensor([0.0500, 0.0500, 0.0000, 0.0000, 0.0000], device='cuda:0')
Latent for 1x4: tensor([0.0500, 0.0500, 0.0000, 0.0000, 0.0000], device='cuda:0')


In [7]:
# Extract meshes
from util.visualization.utils_mesh import get_watertight_mesh_for_latent

mc_resolution = 128
z_base = [0.05, 0.05]  # Sample latent

meshes = {}
for n_studs in n_studs_values:
    z = make_latent(z_base, n_studs, config)
    bounds = problems[n_studs].bounds.to(device)
    
    print(f"Extracting 1x{n_studs} mesh with bounds {bounds.tolist()}...")
    verts, faces = get_watertight_mesh_for_latent(
        netp.f_, netp.params, z, bounds, 
        mc_resolution=mc_resolution, 
        device=device,
        chunks=1, 
        level=0, 
        surpress_watertight=True
    )
    meshes[n_studs] = (verts, faces)
    print(f"  Got {len(verts)} vertices, {len(faces)} faces")

Extracting 1x2 mesh with bounds [[-0.987500011920929, 0.987500011920929], [-0.48750001192092896, 0.48750001192092896], [-0.706250011920929, 0.706250011920929]]...
  Got 70456 vertices, 140908 faces
Extracting 1x4 mesh with bounds [[-1.9874999523162842, 1.9874999523162842], [-0.48750001192092896, 0.48750001192092896], [-0.706250011920929, 0.706250011920929]]...
  Got 32372 vertices, 64740 faces
Extracting 1x5 mesh with bounds [[-2.487499952316284, 2.487499952316284], [-0.48750001192092896, 0.48750001192092896], [-0.706250011920929, 0.706250011920929]]...
  Got 25618 vertices, 51248 faces
Extracting 1x8 mesh with bounds [[-3.987499952316284, 3.987499952316284], [-0.48750001192092896, 0.48750001192092896], [-0.706250011920929, 0.706250011920929]]...
  Got 15489 vertices, 31014 faces
Extracting 1x12 mesh with bounds [[-5.987500190734863, 5.987500190734863], [-0.48750001192092896, 0.48750001192092896], [-0.706250011920929, 0.706250011920929]]...
  Got 10370 vertices, 20732 faces


In [8]:
# Visualize with k3d
import k3d

fig = k3d.plot(height=600)

colors = [0x3498db, 0xe74c3c, 0x2ecc71, 0xf39c12]
offset_x = 0

for i, n_studs in enumerate(n_studs_values):
    verts, faces = meshes[n_studs]
    if len(verts) > 0:
        # Offset for side-by-side display
        verts_display = np.array(verts, dtype=np.float32)
        verts_display[:, 0] += offset_x
        
        fig += k3d.mesh(
            verts_display, 
            np.array(faces, dtype=np.uint32),
            color=colors[i % len(colors)],
            side='double',
            name=f"1x{n_studs}"
        )
        
        # Update offset based on brick width
        offset_x += problems[n_studs].bounds[0, 1].item() * 2.5
    else:
        print(f"WARNING: Empty mesh for 1x{n_studs}")

fig.display()

Output()

In [9]:
# Test multiple latent vectors for one brick type
n_studs = n_studs_values[0]
bounds = problems[n_studs].bounds.to(device)

# Grid of latents
grid_size = 3
z_range = np.linspace(0.0, 0.1, grid_size)

fig2 = k3d.plot(height=600)
spacing = 3.0

for i, z0 in enumerate(z_range):
    for j, z1 in enumerate(z_range):
        z = make_latent([z0, z1], n_studs, config)
        verts, faces = get_watertight_mesh_for_latent(
            netp.f_, netp.params, z, bounds,
            mc_resolution=64, device=device,
            chunks=1, level=0, surpress_watertight=True
        )
        
        if len(verts) > 0:
            verts_display = np.array(verts, dtype=np.float32)
            verts_display[:, 0] += j * spacing
            verts_display[:, 1] += i * spacing
            
            fig2 += k3d.mesh(
                verts_display,
                np.array(faces, dtype=np.uint32),
                color=0x4488ff,
                side='double',
                name=f"z=[{z0:.2f},{z1:.2f}]"
            )

print(f"Latent grid for 1x{n_studs}:")
fig2.display()

Latent grid for 1x2:


Output()

In [10]:
# Mesh statistics
import trimesh

print("Mesh Statistics:")
print("=" * 50)

for n_studs in n_studs_values:
    verts, faces = meshes[n_studs]
    if len(verts) == 0:
        print(f"1x{n_studs}: EMPTY MESH")
        continue
    
    tm = trimesh.Trimesh(vertices=verts, faces=faces)
    n_components = len(tm.split(only_watertight=False))
    
    print(f"\n1x{n_studs}:")
    print(f"  Vertices: {len(verts)}")
    print(f"  Faces: {len(faces)}")
    print(f"  Components: {n_components} {'✓' if n_components == 1 else '✗'}")
    print(f"  Watertight: {tm.is_watertight}")
    print(f"  Volume: {tm.volume:.4f}")

Mesh Statistics:

1x2:
  Vertices: 70456
  Faces: 140908
  Components: 1 ✓
  Watertight: True
  Volume: -2.6519

1x4:
  Vertices: 32372
  Faces: 64740
  Components: 1 ✓
  Watertight: True
  Volume: -5.1013

1x5:
  Vertices: 25618
  Faces: 51248
  Components: 1 ✓
  Watertight: True
  Volume: -6.2888

1x8:
  Vertices: 15489
  Faces: 31014
  Components: 2 ✗
  Watertight: True
  Volume: -9.4514

1x12:
  Vertices: 10370
  Faces: 20732
  Components: 3 ✗
  Watertight: True
  Volume: -12.9648


### Latent space (naked)
No input z — fixed grid over (z0, z1). This is the model's latent space as-is. All brick types from the model config; each block = one brick type, row = z0, column = z1.

In [11]:
# Latent space (naked): no user z — fixed grid over (z0, z1), all brick types from model config
latent_n_studs = list(dict.fromkeys(config.get('n_studs_values', n_studs_values)))
for n in latent_n_studs:
    if n not in problems:
        prob_cfg = {**base_problem_cfg, 'n_studs': n, 'n_studs_y': None, 'height_scale': 1.0, 'no_studs': False}
        sampling = {k: v for k, v in problem_sampling.items() if k != 'problem_str'}
        problems[n] = get_problem(problem_config=prob_cfg, **sampling)

grid_size_full = 5
z_min, z_max = 0.0, 0.15
z_range_full = np.linspace(z_min, z_max, grid_size_full)
# Spacing so meshes don't overlap (bricks ~4–5 units wide)
spacing = 10.0
block_spacing = 25.0

fig_full = k3d.plot(height=800)
for i_n, n_studs in enumerate(latent_n_studs):
    bounds = problems[n_studs].bounds.to(device)
    row_offset = i_n * (grid_size_full * spacing + block_spacing)
    for i, z0 in enumerate(z_range_full):
        for j, z1 in enumerate(z_range_full):
            z = make_latent([z0, z1], n_studs, config)
            verts, faces = get_watertight_mesh_for_latent(
                netp.f_, netp.params, z, bounds,
                mc_resolution=48, device=device,
                chunks=1, level=0, surpress_watertight=True
            )
            if len(verts) > 0:
                verts_display = np.array(verts, dtype=np.float32)
                verts_display[:, 0] += j * spacing
                verts_display[:, 1] += i * spacing + row_offset
                fig_full += k3d.mesh(
                    verts_display,
                    np.array(faces, dtype=np.uint32),
                    color=0x4488ff,
                    side='double',
                    name=f"1x{n_studs} z=[{z0:.2f},{z1:.2f}]"
                )

print("Latent space (naked): no input z. z0,z1 in [%.2f,%.2f], %dx%d per type" % (z_min, z_max, grid_size_full, grid_size_full))
print("Rows (bottom→top):", ", ".join("1x%d" % n for n in latent_n_studs), "— within each row: z0 (↑), z1 (→)")
fig_full.display()

Latent space (naked): no input z. z0,z1 in [0.00,0.15], 5x5 per type
Rows (bottom→top): 1x2, 1x4, 1x5, 1x8, 1x12 — within each row: z0 (↑), z1 (→)


Output()

In [12]:
# Ensure problems cache exists if running this section standalone
if 'problems' not in globals():
    problems = {}


# Experiments

## Latent Interpolation

In [14]:
# Interpolate brick type from 1x2 → 1x4 (conditioning n_norm from 0 to 1)
# Uses a fixed shape latent; only the conditioning (brick length) is interpolated.
n_start, n_end = 2, 4  # 1x2 → 1x4

# Ensure problems exist for bounds
for n in (n_start, n_end):
    if n not in problems:
        prob_cfg = {**base_problem_cfg, 'n_studs': n, 'n_studs_y': None, 'height_scale': 1.0, 'no_studs': False}
        sampling = {k: v for k, v in problem_sampling.items() if k != 'problem_str'}
        problems[n] = get_problem(problem_config=prob_cfg, **sampling)

bounds_2 = problems[n_start].bounds.to(device)
bounds_4 = problems[n_end].bounds.to(device)

# Fixed shape latent (so we only see the effect of brick-type interpolation)
z_base = torch.tensor([0.05, 0.05], dtype=torch.float32, device=device)

# Use [2, 4] for conditioning so n_norm is 0 (1x2) and 1 (1x4) — matches training
_saved = config.get('n_studs_values')
config['n_studs_values'] = [2, 4]

K = 8
sp = 6.0
fig_interp = k3d.plot(height=500)

for k, t in enumerate(np.linspace(0.0, 1.0, K)):
    # Conditioning: n_norm = t (0 → 1x2, 1 → 1x4). n_studs = 2 + t*2 gives n_norm = t when n_studs_values=[2,4]
    n_studs_eff = n_start + t * (n_end - n_start)
    z = make_latent(z_base, n_studs_eff, config)
    # Interpolate bounds so marching cubes domain matches the interpolated brick length
    bounds_t = (1 - t) * bounds_2 + t * bounds_4
    verts, faces = get_watertight_mesh_for_latent(
        netp.f_, netp.params, z, bounds_t,
        mc_resolution=48, device=device,
        chunks=1, level=0, surpress_watertight=True
    )
    if len(verts) == 0:
        continue
    verts_display = np.array(verts, dtype=np.float32)
    verts_display[:, 0] += k * sp
    fig_interp += k3d.mesh(
        verts_display,
        np.array(faces, dtype=np.uint32),
        color=0xff8844,
        side='double',
        name=f"t={t:.2f} (1x{2 + t*2:.1f})"
    )

if _saved is not None:
    config['n_studs_values'] = _saved

print(f"Brick-type interpolation 1x{n_start} → 1x{n_end} (conditioning n_norm 0→1), K={K} steps, fixed z_base={z_base.tolist()}")
fig_interp.display()

Brick-type interpolation 1x2 → 1x4 (conditioning n_norm 0→1), K=8 steps, fixed z_base=[0.05000000074505806, 0.05000000074505806]


Output()

## Constraint satisfaction rate versus baselines

In [15]:
# Constraint-satisfaction baseline: load a specific GINN checkpoint and one-shot a 1x2
# Checkpoint: checkpoints/cond_wire/2026_01_27__21_58_20-bnsrxi2p
import glob
import yaml
from util.misc import get_model, get_problem
from util.visualization.utils_mesh import get_watertight_mesh_for_latent

_run = "checkpoints/cond_wire/2026_01_27__21_58_20-bnsrxi2p"
_run_dir = _run if os.path.isabs(_run) else os.path.normpath(os.path.join("..", _run))
_ckpt_path = glob.glob(os.path.join(_run_dir, "*-model.pt"))
if not _ckpt_path:
    raise FileNotFoundError(f"No *-model.pt in {_run_dir}")
_ckpt_path = _ckpt_path[0]
_config_files = glob.glob(os.path.join(_run_dir, "*-config.yml"))
_config_baseline = yaml.safe_load(open(_config_files[0])) if _config_files else {}

_ckpt_baseline = torch.load(_ckpt_path, map_location="cpu", weights_only=False)
_init_params = _ckpt_baseline.get("init_params", {})
_state = _ckpt_baseline.get("model", _ckpt_baseline.get("state_dict"))

# Build model args from init_params or config
if _init_params:
    _model_args = {k: v for k, v in _init_params.items() if k != "layers"}
    _raw = _init_params.get("layers", [128, 128, 128])
    if len(_raw) >= 2 and (_raw[-1] == 1 or getattr(_raw[-1], "item", None) and _raw[-1].item() == 1):
        _model_args["layers"] = _raw[1:-1]
        _in_dim = int(_raw[0]) if hasattr(_raw[0], "__int__") else _raw[0]
        _tiled = 2 if _model_args.get("use_tiled_coords", False) else 0
        _dist = 1 if (_model_args.get("use_dist_to_edge") and _model_args.get("n_studs_values")) else 0
        _model_args["nz"] = _in_dim - 3 - _tiled - _dist
    else:
        _model_args["layers"] = list(_raw)
    _model_args.setdefault("model_str", "cond_wire")
    _model_args.setdefault("nx", 3)
    for _k in ["first_omega_0", "hidden_omega_0", "scale"]:
        _model_args.pop(_k, None)
    _model_args.setdefault("w0_initial", _config_baseline.get("model", {}).get("w0_initial", 18))
    _model_args.setdefault("w0", _config_baseline.get("model", {}).get("w0", 1.0))
    _model_args.setdefault("wire_scale", _config_baseline.get("model", {}).get("wire_scale", 6))
else:
    _model_args = dict(_config_baseline.get("model", {}))

# LEGO 1x2 for this section: ensure conditioning if model expects it
_config_baseline["problem"] = _config_baseline.get("problem", {})
_config_baseline["problem"]["problem_str"] = "lego_1xN"
_config_baseline["problem"]["n_studs"] = 2
_config_baseline.setdefault("n_studs_values", [2, 4])
_config_baseline.setdefault("condition_on_n_studs", True)
_ps = _config_baseline.get("problem_sampling", {})
for _k, _v in [("nx", 3), ("n_points_domain", 2048), ("n_points_envelope", 8192), ("n_points_interfaces", 4096), ("n_points_normals", 4096)]:
    _ps.setdefault(_k, _v)

_model_baseline = get_model(**_model_args)
_model_baseline.load_state_dict(_state)
_model_baseline.eval()
_model_baseline.to(device)
_netp_baseline = NetWithPartials.create_from_model(_model_baseline, nz=_model_args["nz"], nx=_model_args["nx"])
_netp_baseline.params = {k: v.to(device) for k, v in _netp_baseline.params.items()}

_prob_cfg = {**_config_baseline["problem"], "n_studs": 2, "n_studs_y": None, "height_scale": 1.0, "no_studs": False}
_problem_1x2 = get_problem(problem_config=_prob_cfg, **{k: v for k, v in _ps.items() if k != "problem_str"})
_bounds_1x2 = _problem_1x2.bounds.to(device)

# One-shot: sample z from N(0, I) and decode to 1x2
# z must have exactly _model_args["nz"] dims (this checkpoint has nz=2, unconditional)
z_one = torch.randn(_model_args["nz"], device=device, dtype=torch.float32) * 0.1
if _model_args["nz"] >= 3 and _config_baseline.get("condition_on_n_studs"):
    _n_vals = _config_baseline.get("n_studs_values", [2, 4])
    _n_norm = (2 - min(_n_vals)) / max(max(_n_vals) - min(_n_vals), 1)
    z_one = torch.cat([z_one[: _model_args["nz"] - 1], torch.tensor([_n_norm], device=device, dtype=torch.float32)])

verts_1x2, faces_1x2 = get_watertight_mesh_for_latent(
    _netp_baseline.f_, _netp_baseline.params, z_one, _bounds_1x2,
    mc_resolution=64, device=device, chunks=1, level=0, surpress_watertight=True
)
print(f"Loaded {_ckpt_path}")
print(f"One-shot 1x2: z ~ N(0,I) scaled 0.1 → {len(verts_1x2)} vertices, {len(faces_1x2)} faces")

fig_one = k3d.plot(height=400)
if len(verts_1x2) > 0:
    fig_one += k3d.mesh(np.array(verts_1x2, dtype=np.float32), np.array(faces_1x2, dtype=np.uint32), color=0x3498db, side="double", name="1x2 one-shot")
fig_one.display()

Created LEGO 1x2 problem (height_scale=1.00):
  Bounds: [[-0.987500011920929, 0.987500011920929], [-0.48750001192092896, 0.48750001192092896], [-0.706250011920929, 0.706250011920929]]
  Stud centers: 2 studs
  Points: 16383 far_outside, 32766 outside, 16383 around_if, 4096 inside, 4094 interface, 6 walls


RuntimeError: mat1 and mat2 shapes cannot be multiplied (89280x6 and 5x128)